<a href="https://colab.research.google.com/github/SithmiDesilva/Hallucination-Aware-AI-Framework-for-Trustworthy-Supply-Chain-Decision-Intelligence.-/blob/News-data-preprocessing/News_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu

In [2]:
import os, re, json, time, hashlib
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/IRP/news_data'
except Exception:
    BASE = 'news_data'

CORPUS_RUN_ID = '20261125T132210Z'          # the frozen collection run
PREP_DIR = os.path.join(BASE, f'preprocessed_{CORPUS_RUN_ID}')
os.makedirs(PREP_DIR, exist_ok=True)

# Must match the collection notebook
DATASET_WINDOWS = {
    'M5':     ('2011-01-29', '2016-06-19'),
    'DataCo': ('2015-01-01', '2018-01-31'),
}

raw = pd.read_json(os.path.join(BASE, f'news_articles_{CORPUS_RUN_ID}.jsonl'),
                   lines=True, convert_dates=False)
raw['doc_id'] = raw['url'].map(lambda u: hashlib.md5(u.encode()).hexdigest()[:12])
print(f'Loaded {len(raw)} articles | keep_candidate: {raw["keep_candidate"].sum()}')
print('Outputs go to:', PREP_DIR)

Mounted at /content/drive
Loaded 1228 articles | keep_candidate: 1155
Outputs go to: /content/drive/MyDrive/IRP/news_data/preprocessed_20261125T132210Z


In [3]:
MAX_CHARS = 15000      # cap very long documents (reports, PDFs)
MIN_CHARS = 800        # drop pages that are too short after cleaning

MD_IMAGE = re.compile(r'!\[[^\]]*\]\([^)]*\)')
MD_LINK = re.compile(r'\[([^\]]*)\]\([^)]*\)')
URL_RE = re.compile(r'https?://\S+')
HTML_TAG = re.compile(r'<[^>]+>')
JUNK_LINE = re.compile(r'^\s*(share|tweet|email|print|advertisement|sign up|subscribe|follow us|read more'
                       r'|click here|cookie|privacy policy|terms of (use|service)|all rights reserved|©|copyright)\b', re.I)
STOP_SECTION = re.compile(r'^\s*#*\s*(related (articles|stories|content|posts)|more from|recommended( for you)?'
                          r'|you may also like|comments)\s*:?\s*$', re.I)

def clean_text(t, max_chars=MAX_CHARS):
    t = MD_IMAGE.sub(' ', t or '')
    t = MD_LINK.sub(r'\1', t)
    t = URL_RE.sub(' ', t)
    t = HTML_TAG.sub(' ', t)
    out = []
    for line in t.splitlines():
        s = line.strip()
        if STOP_SECTION.match(s) and len(out) > 5:
            break                                   # cut "Related articles" and everything after
        if not s:
            out.append('')
            continue
        if JUNK_LINE.match(s) and len(s) < 120:
            continue                                # share buttons, footers
        if len(s.split()) <= 3 and not re.search(r'[.!?:]$', s) and not s.startswith('#'):
            continue                                # menu items / nav links
        out.append(s)
    t = '\n'.join(out)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[ \t]{2,}', ' ', t).strip()
    if len(t) > max_chars:                          # cut at the last full sentence
        cut = t[:max_chars]
        t = cut[:cut.rfind('.') + 1] if cut.rfind('.') > max_chars * 0.8 else cut
    return t

ACADEMIC = ('doi.org', 'journals.', 'sciencedirect', 'springer', 'wiley', 'tandfonline', 'mdpi', 'arxiv',
            'researchgate', 'digitalcommons', 'ssrn', 'jstor', 'nature.com', 'frontiersin', 'academia.edu',
            'ieeexplore', 'scholar.')
PRESS = ('prnewswire', 'businesswire', 'globenewswire', 'accesswire', 'newswire')

def source_type(domain):
    d = domain or ''
    if any(k in d for k in ACADEMIC):
        return 'academic'
    if any(k in d for k in PRESS):
        return 'press_release'
    if d.endswith('.gov') or '.gov.' in d or 'federalregister' in d:
        return 'government'
    return 'news_or_other'

docs = raw[raw['keep_candidate']].copy()
docs['clean_text'] = docs['text'].map(clean_text)
docs['clean_len'] = docs['clean_text'].str.len()
docs['source_type'] = docs['domain'].map(source_type)

# near-duplicate check on the opening of the cleaned text (syndicated copies)
docs['dup_key'] = docs['clean_text'].str.lower().str.replace(r'\W+', ' ', regex=True).str[:1500].map(
    lambda s: hashlib.md5(s.encode()).hexdigest())
docs['flag_dup_text'] = docs['dup_key'].duplicated(keep='first')
docs['flag_too_short'] = docs['clean_len'] < MIN_CHARS

before = len(docs)
docs = docs[~docs['flag_dup_text'] & ~docs['flag_too_short']].reset_index(drop=True)
print(f'{before} -> {len(docs)} docs after cleaning (dropped dups/too short: {before - len(docs)})')
print(docs['source_type'].value_counts())
print(docs['clean_len'].describe(percentiles=[.5, .9, .99]).round(0))

1155 -> 1149 docs after cleaning (dropped dups/too short: 6)
source_type
news_or_other    1004
government         64
press_release      45
academic           36
Name: count, dtype: int64
count     1149.0
mean      4816.0
std       4014.0
min        812.0
50%       3540.0
90%      12381.0
99%      14983.0
max      15000.0
Name: clean_len, dtype: float64


In [4]:
# Export 50 random docs for YOU to label by hand. Used later to measure LLM accuracy.
TODO_PATH = os.path.join(PREP_DIR, 'manual_labels_TODO.csv')
DONE_PATH = os.path.join(PREP_DIR, 'manual_labels_done.csv')

if not os.path.exists(TODO_PATH):          # never overwrite once created
    sample = docs.sample(min(50, len(docs)), random_state=42)
    sample = sample[['doc_id', 'title', 'domain', 'publishedDate', 'url']].assign(
        text_preview=sample['clean_text'].str[:600],
        manual_relevant='',        # fill: y / n
        manual_event_type='',      # fill: one of EVENT_TYPES (see cell 6)
        notes='')
    sample.to_csv(TODO_PATH, index=False)
    print('Created', TODO_PATH)
else:
    print('Already exists (not overwritten):', TODO_PATH)
print('Fill it in Google Sheets/Excel, then save it as:', DONE_PATH)

Created /content/drive/MyDrive/IRP/news_data/preprocessed_20261125T132210Z/manual_labels_TODO.csv
Fill it in Google Sheets/Excel, then save it as: /content/drive/MyDrive/IRP/news_data/preprocessed_20261125T132210Z/manual_labels_done.csv
